# CS156 Assignment #1: Implementation

Code for the reminder classifier. The write-up lives in `write-up.ipynb`.

The Telegram export and the CSVs contain personal messages, so they live in `data/` and are gitignored. 

## 1. Data

Turns the Telegram Desktop "Export chat history" HTML into a CSV with one row per unique message I sent:

- `id`: SHA-256 hash of the message text
- `message`: the raw text
- `has_remind_keyword`: 1 if the text matches `\bremind\w*`, else 0
- `timestamp`: ISO 8601
- `label`: empty until the labeling step (reminder = 1, other = 0)

Messages from the first days (assistant setup) and messages that are only a filler reply ("yes", "lgtm") are dropped.

In [1]:
import csv
import hashlib
import re
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

USER_NAME = "Stiven Triana"
REMIND_PATTERN = re.compile(r"\bremind\w*", re.IGNORECASE)
TIMESTAMP_FORMAT = "%d %B %Y, %H:%M:%S"
CSV_COLUMNS = ["id", "message", "has_remind_keyword", "timestamp", "label"]
SETUP_DAYS = 2
NON_WORD_PATTERN = re.compile(r"[^\w\s]")
FILLER_PHRASES = frozenset(
    [
        "yes", "no", "y", "n", "yep", "yeah", "yup", "nope", "nah",
        "ok", "okay", "k", "kk", "sure", "go ahead", "do it", "sounds good",
        "lgtm", "thanks", "thank you", "thx", "ty", "cool", "nice", "great",
        "perfect", "done", "continue", "proceed", "correct", "right",
        "fine", "approved", "confirmed", "hey", "hi", "hello",
    ]
)
HTML_PATH = Path("data/messages.html")
CSV_PATH = Path("data/messages.csv")


In [2]:
def extract_text(text_div) -> str:
    for line_break in text_div.find_all("br"):
        line_break.replace_with("\n")
    return text_div.get_text().strip()


def parse_timestamp(date_div) -> str:
    raw = date_div.get("title", "") if date_div else ""
    try:
        return datetime.strptime(raw, TIMESTAMP_FORMAT).isoformat()
    except ValueError:
        return ""


def extract_user_messages(html_path: Path) -> list[dict]:
    soup = BeautifulSoup(html_path.read_text(encoding="utf-8"), "html.parser")
    messages = []
    current_sender = ""

    for message_div in soup.select("div.message.default"):
        from_name_div = message_div.select_one("div.from_name")
        # "joined" messages omit the sender and inherit it from the previous one
        if from_name_div:
            current_sender = from_name_div.get_text().strip()
        if current_sender != USER_NAME:
            continue

        text_div = message_div.select_one("div.body > div.text")
        if not text_div:
            continue

        text = extract_text(text_div)
        if not text:
            continue

        messages.append(
            {
                "message": text,
                "timestamp": parse_timestamp(message_div.select_one("div.date")),
            }
        )
    return messages


def drop_setup_period(messages: list[dict]) -> list[dict]:
    timestamps = [
        datetime.fromisoformat(item["timestamp"])
        for item in messages
        if item["timestamp"]
    ]
    if not timestamps:
        return messages

    cutoff = min(timestamps) + timedelta(days=SETUP_DAYS)
    return [
        item
        for item in messages
        if not item["timestamp"] or datetime.fromisoformat(item["timestamp"]) >= cutoff
    ]


def is_filler(text: str) -> bool:
    normalized = " ".join(NON_WORD_PATTERN.sub(" ", text.lower()).split())
    return normalized in FILLER_PHRASES


def drop_filler(messages: list[dict]) -> list[dict]:
    return [item for item in messages if not is_filler(item["message"])]


def build_rows(messages: list[dict]) -> list[dict]:
    rows = []
    seen_ids = set()
    for item in messages:
        message_id = hashlib.sha256(item["message"].encode("utf-8")).hexdigest()
        if message_id in seen_ids:
            continue
        seen_ids.add(message_id)
        rows.append(
            {
                "id": message_id,
                "message": item["message"],
                "has_remind_keyword": int(bool(REMIND_PATTERN.search(item["message"]))),
                "timestamp": item["timestamp"],
                "label": "",
            }
        )
    return rows


In [3]:
all_messages = extract_user_messages(HTML_PATH)
after_setup = drop_setup_period(all_messages)
after_filler = drop_filler(after_setup)
rows = build_rows(after_filler)

pd.DataFrame(rows, columns=CSV_COLUMNS).to_csv(CSV_PATH, index=False)

print(f"user messages: {len(all_messages)}")
print(f"after dropping first {SETUP_DAYS} days: {len(after_setup)}")
print(f"after dropping filler replies: {len(after_filler)}")
print(f"after dedup: {len(rows)}")
print(f"has_remind_keyword=1: {sum(row['has_remind_keyword'] for row in rows)}")


user messages: 679
after dropping first 2 days: 563
after dropping filler replies: 559
after dedup: 558
has_remind_keyword=1: 238


## 2. Labelling

Labels come from a local LLM served by Ollama (the messages never leave this machine). Each message is sent on its own with temperature 0 and a JSON schema that only allows `{"label": 0}` or `{"label": 1}`. If the reply can't be parsed, the request is retried up to `MAX_ATTEMPTS` times; messages that still fail keep an empty label so a re-run picks them up. The CSV is rewritten after every message, and rows that already have a label are skipped, so the run can be interrupted and resumed.

Requires Ollama running locally with the model pulled (`ollama pull <OLLAMA_MODEL>`). The examples in the prompt are invented, not taken from my messages.

In [ ]:
import json

import requests

OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5:7b-instruct"
LABEL_TEMPERATURE = 0
LABEL_SEED = 156
MAX_ATTEMPTS = 3
REQUEST_TIMEOUT_SECONDS = 120
PROGRESS_EVERY = 50
LABEL_SCHEMA = {
    "type": "object",
    "properties": {"label": {"type": "integer", "enum": [0, 1]}},
    "required": ["label"],
}
LABELING_PROMPT = """You classify messages that a user sent to their personal AI assistant.

Answer 1 if the message is a reminder request, otherwise 0.

A reminder request asks the assistant to remind the user of something at a
specific time or after some delay, once or on a recurring schedule. Messages
that create, change, list or cancel such reminders also count as 1.

Everything else is 0: questions, coding or writing tasks, general chat,
scheduling that is not a reminder, and messages that merely mention the word
"remind" without asking for one.

Reply only with JSON of the form {"label": 0} or {"label": 1}.

Examples:
"remind me to call the dentist tomorrow at 9am" -> {"label": 1}
"every Sunday evening ping me to water the plants" -> {"label": 1}
"cancel my reminder about the gym" -> {"label": 1}
"what reminders do I have set for this week?" -> {"label": 1}
"in 20 minutes tell me the pasta is ready" -> {"label": 1}
"summarize this article about solar panels" -> {"label": 0}
"that reminds me of a song, who sings it?" -> {"label": 0}
"book a table for two on Friday at 8pm" -> {"label": 0}
"write a python function that reverses a list" -> {"label": 0}
"""

In [ ]:
def check_ollama_ready() -> str:
    """Return an empty string if Ollama is up and has the model, else the problem."""
    try:
        response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()
    except requests.RequestException as error:
        return f"Ollama is not reachable at {OLLAMA_URL}: {type(error).__name__}"

    installed = [model["name"] for model in response.json().get("models", [])]
    if OLLAMA_MODEL not in installed:
        return f"model {OLLAMA_MODEL} is not pulled (run: ollama pull {OLLAMA_MODEL})"
    return ""


def ask_model(message: str) -> str | None:
    payload = {
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": LABELING_PROMPT},
            {"role": "user", "content": message},
        ],
        "format": LABEL_SCHEMA,
        "stream": False,
        "options": {"temperature": LABEL_TEMPERATURE, "seed": LABEL_SEED},
    }
    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/chat", json=payload, timeout=REQUEST_TIMEOUT_SECONDS
        )
        response.raise_for_status()
    except requests.RequestException:
        return None
    return response.json().get("message", {}).get("content")


def parse_label(raw_reply: str | None) -> int | None:
    if raw_reply is None:
        return None
    try:
        parsed = json.loads(raw_reply)
    except json.JSONDecodeError:
        return None
    if not isinstance(parsed, dict):
        return None

    label = parsed.get("label")
    if label not in (0, 1) or isinstance(label, bool):
        return None
    return label


def label_message(message: str) -> int | None:
    for _ in range(MAX_ATTEMPTS):
        label = parse_label(ask_model(message))
        if label is not None:
            return label
    return None


def label_all_messages(csv_path: Path) -> int:
    """Label every unlabeled row, saving after each one. Returns the failure count."""
    dataset = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    pending = dataset.index[dataset["label"] == ""]
    failures = 0

    for position, row_index in enumerate(pending, start=1):
        label = label_message(dataset.at[row_index, "message"])
        if label is None:
            failures += 1
            continue

        dataset.at[row_index, "label"] = str(label)
        dataset.to_csv(csv_path, index=False)
        if position % PROGRESS_EVERY == 0:
            print(f"labelled {position}/{len(pending)}")
    return failures

In [ ]:
problem = check_ollama_ready()
if problem:
    print(problem)
else:
    failed = label_all_messages(CSV_PATH)
    labels = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)["label"]
    print(f"model: {OLLAMA_MODEL}")
    print(f"label=1: {(labels == '1').sum()}, label=0: {(labels == '0').sum()}")
    print(f"unlabelled after {MAX_ATTEMPTS} attempts each: {failed}")